# Active BioT5 Diverse Beam Local Collection Review

This notebook follows the active `data_collection/biot5_collection.py` and `data_collection/biot5_generation.py` flow without running the collection script.
It reviews a small ChEBI slice with the active BioT5 collection model, compares the maintained diverse-beam presets, decodes SELFIES first, and prints timing plus molecule-validity and similarity summaries.


## Notes

- This is a local review notebook, not a Kaggle notebook.
- It uses the active collection tokenizer/model path rather than the native `biot5-plus-base-chebi20` review checkpoint.
- The prompt contract is the active text2mol contract: description in, SELFIES out.
- The notebook decodes SELFIES first, then converts valid outputs to SMILES for similarity review.
- `filter_selfies(...)` is kept only as a diagnostic recovery fallback when the decoded text contains noisy extra characters.


In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import selfies
import torch
import transformers

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
assert (PROJECT_ROOT / "src").exists(), "Run this notebook from the Thesis repo root or from notebooks/."

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data_collection import BioT5DiverseBeamGenerator, resolve_biot5_collection_config_paths
from molecules.collection.filtering import CollectionMetricConfig, prepare_reference_groups
from notebooks.biot5_collection_review_support import (
    assess_biot5_native_generation_output,
    summarize_biot5_native_review_records,
)
from src.io_utils import load_yaml, read_jsonl
from src.prompting import build_text2mol_prompt


def print_section(title: str) -> None:
    print(f"\n=== {title} ===")


def print_json(title: str, payload) -> None:
    print_section(title)
    print(json.dumps(payload, indent=2, ensure_ascii=False))


def print_records(title: str, records, *, limit: int | None = None, keys: list[str] | None = None) -> None:
    payload = list(records)
    total = len(payload)
    if keys is not None:
        payload = [{key: row.get(key) for key in keys} for row in payload]
    shown = payload if limit is None else payload[:limit]
    print_section(f"{title} (showing {len(shown)} of {total})")
    print(json.dumps(shown, indent=2, ensure_ascii=False))


version_report = {
    "project_root": str(PROJECT_ROOT),
    "python": sys.version.split()[0],
    "selfies": selfies.__version__,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
}
print_json("Environment", version_report)


In [ ]:
ACTIVE_CONFIG_PATH = PROJECT_ROOT / "configs" / "collect_biot5_chebi20.yaml"
ACTIVE_CONFIG = resolve_biot5_collection_config_paths(
    load_yaml(ACTIVE_CONFIG_PATH),
    project_root=PROJECT_ROOT,
)

TRAIN_FILE = Path(ACTIVE_CONFIG["data"]["train_file"])
MODEL_NAME_OR_PATH = str(ACTIVE_CONFIG["model"]["model_name_or_path"])
MODEL_MAX_LENGTH = int(
    ACTIVE_CONFIG["model"].get(
        "model_max_length",
        ACTIVE_CONFIG["generation"].get("max_source_length", ACTIVE_CONFIG["generation"].get("max_length", 512)),
    )
)
DEVICE = str(ACTIVE_CONFIG["model"].get("device", "auto"))

SELECTED_DESCRIPTION_IDS = ["129626631", "12699"]
NUM_SAMPLES = 30
ALLOW_FILTER_FALLBACK = True

ACTIVE_DEFAULT_CONFIG = dict(ACTIVE_CONFIG["generation"])
ACTIVE_DEFAULT_CONFIG.setdefault("max_length", MODEL_MAX_LENGTH)
ACTIVE_DEFAULT_CONFIG["target_count"] = NUM_SAMPLES
ACTIVE_DEFAULT_CONFIG["target_molecules_per_description"] = NUM_SAMPLES
ACTIVE_DEFAULT_CONFIG["num_return_sequences"] = NUM_SAMPLES
ACTIVE_DEFAULT_CONFIG["num_beams"] = max(int(ACTIVE_DEFAULT_CONFIG.get("num_beams", NUM_SAMPLES)), NUM_SAMPLES)
if "num_beam_groups" in ACTIVE_DEFAULT_CONFIG:
    num_beam_groups = int(ACTIVE_DEFAULT_CONFIG["num_beam_groups"])
    if ACTIVE_DEFAULT_CONFIG["num_beams"] % num_beam_groups != 0:
        raise ValueError(
            "Notebook 00 requires num_beams to be divisible by num_beam_groups; "
            f"got {ACTIVE_DEFAULT_CONFIG['num_beams']} and {num_beam_groups}."
        )

DIVERSE_BEAM_FAST_CONFIG = dict(ACTIVE_DEFAULT_CONFIG)
DIVERSE_BEAM_FAST_CONFIG["num_beam_groups"] = 5
DIVERSE_BEAM_FAST_CONFIG["diversity_penalty"] = 0.3

GENERATION_CONFIGS = {
    "active_default": ACTIVE_DEFAULT_CONFIG,
    "diverse_beam_fast": DIVERSE_BEAM_FAST_CONFIG,
}

METRIC_CONFIG = CollectionMetricConfig(
    fingerprint_radius=int(ACTIVE_CONFIG["filtering"]["fingerprint_radius"]),
    fingerprint_num_bits=int(ACTIVE_CONFIG["filtering"]["fingerprint_num_bits"]),
    acceptance_dice_threshold=float(ACTIVE_CONFIG["filtering"]["acceptance_dice_threshold"]),
)

config_preview = {
    "active_config_path": str(ACTIVE_CONFIG_PATH),
    "train_file": str(TRAIN_FILE),
    "model_name_or_path": MODEL_NAME_OR_PATH,
    "model_max_length": MODEL_MAX_LENGTH,
    "device": DEVICE,
    "selected_description_ids": SELECTED_DESCRIPTION_IDS,
    "num_samples": NUM_SAMPLES,
    "allow_filter_fallback": ALLOW_FILTER_FALLBACK,
    "generation_configs": GENERATION_CONFIGS,
}
print_json("Notebook Config", config_preview)


In [ ]:
all_train_records = read_jsonl(TRAIN_FILE)
records_by_id = {str(record.get("id")): record for record in all_train_records}
selected_records = [
    records_by_id[record_id]
    for record_id in SELECTED_DESCRIPTION_IDS
    if record_id in records_by_id
]
missing_ids = [record_id for record_id in SELECTED_DESCRIPTION_IDS if record_id not in records_by_id]
if missing_ids:
    print("Warning: missing description IDs:", missing_ids)
if not selected_records:
    raise ValueError(f"None of the requested description IDs were found: {SELECTED_DESCRIPTION_IDS}")

reference_groups = prepare_reference_groups(all_train_records, METRIC_CONFIG)

selected_preview = [
    {
        "id": str(record.get("id")),
        "description": str(record.get("description")),
        "reference_selfies": str(record.get("selfies")),
        "reference_smiles": str(record.get("source_smiles")),
    }
    for record in selected_records
]
print_records("Selected descriptions", selected_preview)


In [ ]:
prompt_by_id = {}
for record in selected_records:
    description_id = str(record.get("id"))
    description = str(record.get("description"))
    prompt_by_id[description_id] = build_text2mol_prompt(description)

prompt_preview = [
    {
        "description_id": str(record.get("id")),
        "prompt_variant": "active_text2mol",
        "prompt_text": prompt_by_id[str(record.get("id"))],
    }
    for record in selected_records
]
print_records("Prompt preview", prompt_preview)


In [ ]:
initial_generation_config_name = next(iter(GENERATION_CONFIGS))
generator = BioT5DiverseBeamGenerator(
    model_name_or_path=MODEL_NAME_OR_PATH,
    device_name=DEVICE,
    model_max_length=MODEL_MAX_LENGTH,
    generation_config=GENERATION_CONFIGS[initial_generation_config_name],
)

generator_preview = {
    "device": str(generator.device),
    "supports_remote_group_beam_search": bool(generator.supports_remote_group_beam_search),
    "num_selected_descriptions": len(selected_records),
    "num_samples_per_prompt": NUM_SAMPLES,
    "generation_config_names": list(GENERATION_CONFIGS.keys()),
}
print_json("Generator Preview", generator_preview)


In [ ]:
raw_generations = []
for generation_config_name, generation_config in GENERATION_CONFIGS.items():
    generator.generation_config = dict(generation_config)
    print_section(f"Running {generation_config_name}")
    for record in selected_records:
        description_id = str(record.get("id"))
        description = str(record.get("description"))
        prompt_text = prompt_by_id[description_id]
        target_count = int(generation_config["target_count"])
        started_at = time.perf_counter()
        outputs = generator.generate_candidates(prompt_text, target_count)
        elapsed_seconds_batch = time.perf_counter() - started_at
        seconds_per_sample_batch = elapsed_seconds_batch / max(len(outputs), 1)
        print(
            f"{generation_config_name} | {description_id} | active_text2mol | "
            f"samples={len(outputs)} | elapsed_seconds={elapsed_seconds_batch:.2f} | "
            f"seconds_per_sample={seconds_per_sample_batch:.3f}"
        )
        for candidate_index, raw_prediction_text in enumerate(outputs):
            raw_generations.append(
                {
                    "generation_config_name": generation_config_name,
                    "description_id": description_id,
                    "description": description,
                    "prompt_variant": "active_text2mol",
                    "candidate_index": candidate_index,
                    "raw_prediction_text": raw_prediction_text,
                    "elapsed_seconds_batch": elapsed_seconds_batch,
                    "seconds_per_sample_batch": seconds_per_sample_batch,
                }
            )

generation_report = {
    "num_rows": len(raw_generations),
    "description_ids": sorted({row["description_id"] for row in raw_generations}),
    "prompt_variants": sorted({row["prompt_variant"] for row in raw_generations}),
    "generation_configs": sorted({row["generation_config_name"] for row in raw_generations}),
}
print_json("Generation Report", generation_report)
print_records(
    "Raw generation preview",
    raw_generations,
    limit=12,
    keys=[
        "generation_config_name",
        "description_id",
        "prompt_variant",
        "candidate_index",
        "raw_prediction_text",
    ],
)


In [ ]:
review_rows = []
for row in raw_generations:
    review_rows.append(
        assess_biot5_native_generation_output(
            description_id=row["description_id"],
            description=row["description"],
            prompt_variant=row["prompt_variant"],
            candidate_index=row["candidate_index"],
            raw_prediction_text=row["raw_prediction_text"],
            references=reference_groups[row["description"]],
            metric_config=METRIC_CONFIG,
            generation_config_name=row["generation_config_name"],
            elapsed_seconds_batch=row["elapsed_seconds_batch"],
            seconds_per_sample_batch=row["seconds_per_sample_batch"],
            allow_filter_fallback=ALLOW_FILTER_FALLBACK,
        )
    )

detail_columns = [
    "generation_config_name",
    "description_id",
    "prompt_variant",
    "candidate_index",
    "cleaned_selfies",
    "parsed_selfies",
    "selected_selfies",
    "filtered_selfies",
    "used_filter_selfies_fallback",
    "decoded_smiles",
    "canonical_smiles",
    "derived_selfies",
    "is_valid_selfies",
    "is_valid_smiles",
    "best_reference_smiles",
    "max_dice_similarity",
    "passes_similarity_threshold",
    "rejection_reason",
]
print_records("Assessment preview", review_rows, limit=30, keys=detail_columns)


In [ ]:
summary_rows = summarize_biot5_native_review_records(review_rows)
print_records("Summary rows", summary_rows)

print_section("Compact Summary")
for summary in summary_rows:
    print(
        f"{summary['generation_config_name']} | {summary['description_id']} | {summary['prompt_variant']} | "
        f"elapsed={summary['elapsed_seconds_batch']:.2f}s | "
        f"sec_per_sample={summary['seconds_per_sample_batch']:.3f} | "
        f"valid_selfies={summary['valid_selfies_rate']:.3f} | "
        f"filter_recovery={summary['filter_selfies_recovery_rate']:.3f} | "
        f"valid_smiles={summary['valid_smiles_rate']:.3f} | "
        f"unique={summary['unique_canonical_smiles_count']} | "
        f"avg_dice={summary['avg_max_dice_similarity']:.3f} | "
        f"best_dice={summary['best_max_dice_similarity']:.3f}"
    )


## Review Questions

After the notebook runs, use the summary and assessment preview to answer:

1. Does the active diverse-beam default produce decodable SELFIES reliably on these descriptions?
2. How often is `filter_selfies(...)` needed to recover otherwise valid active-pipeline outputs?
3. Does the faster diverse-beam preset preserve usable molecule diversity and similarity quality?
4. Is the active collection configuration stable enough for broader ChEBI collection runs?
